In [4]:
import pandas as pd
import numpy as np

df_panel = pd.read_csv(r"C:\Users\Ozoha\Documents\.ipynb_checkpoints\credit-risk-ews\data\firm_year_panel.csv")
df_panel.columns.tolist()

['SP_ENTITY_ID',
 'SP_COMPANY_NAME',
 'SP_TICKER',
 'fiscal_year',
 'iq_cash_equiv',
 'iq_cash_oper',
 'iq_ebit',
 'iq_ebitda',
 'iq_interest_exp',
 'iq_lt_debt',
 'iq_st_debt',
 'iq_total_assets',
 'iq_total_rev',
 'pcd_total_liab']

In [6]:
df_panel["total_debt"] = df_panel["iq_lt_debt"] + df_panel["iq_st_debt"]

#Leverage Ratio

In [7]:
df_panel["leverage"] = df_panel["total_debt"] / df_panel["iq_total_assets"]

#Interest Coverage Ratio

In [8]:
df_panel["interest_coverage"] = df_panel["iq_ebit"] / df_panel["iq_interest_exp"]

#Profitability (ROA-style)

In [9]:
df_panel["roa"] = df_panel["iq_ebit"] / df_panel["iq_total_assets"]

#Cash-Flow Coverage

In [11]:
df_panel["ocf_to_debt"] = df_panel["iq_cash_oper"] / df_panel["total_debt"]

#Handling Pathological Values

In [16]:
df_panel.loc[
    df_panel["iq_interest_exp"] <= 0,
    "interest_coverage"
] = np.nan
df_panel.loc[
    df_panel["total_debt"] <= 0,
    "ocf_to_debt"
] = np.nan

In [17]:
df_panel[["leverage", "interest_coverage", "roa", "ocf_to_debt"]].describe(percentiles = [0.01, 0.05, 0.95, 0.99])

,leverage,interest_coverage,roa,ocf_to_debt
count,2341.000000,5.000000,1.028100e+04,2345.000000
mean,-15.312803,31.479664,-inf,-0.178298
std,844.961932,55.524337,NaN,4.807285
min,-40864.000000,-11.807862,-inf,-88.840560
1%,0.010756,-11.272906,-4.515272e+01,-11.278975
5%,0.057581,-9.133082,-2.626822e+00,-2.347095
50%,0.327734,8.529412,3.829892e-02,0.135894
95%,2.599728,107.742319,2.295783e-01,1.139897
99%,19.197756,122.775360,5.117225e-01,5.586654
max,725.326875,126.533621,6.296000e+03,119.666667


#Clean and Winsorize Ratios

In [18]:
bad_assets = df_panel["iq_total_assets"] <= 0
df_panel.loc[bad_assets, ["leverage", "roa"]] = np.nan

In [19]:
def winsorize(series, lower=0.01, upper=0.99):
    return series.clip(
        lower=series.quantile(lower),
        upper=series.quantile(upper))

In [20]:
df_panel["leverage_w"] = winsorize(df_panel["leverage"])
df_panel["interest_coverage_w"] = winsorize(df_panel["interest_coverage"])
df_panel["roa_w"] = winsorize(df_panel["roa"])
df_panel["ocf_to_debt_w"] = winsorize(df_panel["ocf_to_debt"])

In [21]:
df_panel[
    ["leverage_w", "interest_coverage_w", "roa_w", "ocf_to_debt_w"]
].describe(percentiles=[0.01, 0.05, 0.95, 0.99])

,leverage_w,interest_coverage_w,roa_w,ocf_to_debt_w
count,2340.000000,5.000000,10277.000000,2345.000000
mean,0.802028,30.835003,-0.946822,-0.177317
std,2.315535,53.812991,5.183654,1.845411
min,0.011357,-11.272906,-44.359724,-11.278975
1%,0.011671,-10.759348,-44.208716,-10.964875
5%,0.057599,-8.705117,-2.622849,-2.347095
50%,0.327926,8.529412,0.038299,0.135894
95%,2.599763,104.735711,0.229193,1.139897
99%,19.088492,119.167430,0.506036,5.499481
max,19.202550,122.775360,0.506054,5.586654


In [23]:
df_panel = df_panel.sort_values(["SP_ENTITY_ID", "fiscal_year"]).reset_index(drop = True)

#Defining Distress

In [39]:
df_panel["distress"] = ((df_panel["interest_coverage_w"] < 1) |
                       (df_panel["iq_interest_exp"].isna())
                       ).astype(int)

In [40]:
df_panel["distress"].value_counts(normalize = True)

distress
0    0.857635
1    0.142365
Name: proportion, dtype: float64

In [41]:
df_panel["default_t_plus_1"] = df_panel.groupby("SP_ENTITY_ID")["distress"].shift(-1)

In [42]:
df_model = df_panel.dropna(subset = ["default_t_plus_1"]).copy()
df_model["default_t_plus_1"] = df_model["default_t_plus_1"].astype(int)

In [43]:
df_model["default_t_plus_1"].value_counts(normalize = True)

default_t_plus_1
0    0.863227
1    0.136773
Name: proportion, dtype: float64

In [45]:
df_model.groupby("default_t_plus_1")[
    ["interest_coverage_w", "leverage_w", "roa_w"]
].median()

,interest_coverage_w,leverage_w,roa_w
default_t_plus_1,,,
0,8.529412,0.328158,0.043624
1,NaN,0.228344,-0.058299


In [46]:
df_panel.to_csv(r"C:\Users\Ozoha\Documents\.ipynb_checkpoints\credit-risk-ews\data\data_model_firm.csv", index = False)